# 라이브러리 및 데이터 불러오기

In [100]:
import pymysql
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

conn = pymysql.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

print("DB 연결 성공!")

DB 연결 성공!


In [3]:
query = """
SELECT *
FROM hackle_events
;
"""

df = pd.read_sql(query, conn)

df.head()

C:\Users\kkw53\AppData\Local\Temp\ipykernel_11600\1316781957.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,event_id,event_datetime,event_key,session_id,id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,2023-07-18 19:40:17,$session_start,4OzYh3seq3VKytpSn5pvQkZNQii1,00000533-3f1c-4b3b-81f1-0c8f35754b4e,,,NaN,NaN,NaN,NaN
1,00000716-27e9-4e72-a602-d0ce61784b06,2023-07-18 21:07:24,click_question_open,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,00000716-27e9-4e72-a602-d0ce61784b06,,,64.0,436.0,4830.0,NaN
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,2023-08-06 20:18:03,click_bottom_navigation_profile,6bcea65d-9f40-46fc-888c-700fe707483f,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,,,26.0,174.0,4729.0,NaN
3,00000981-5e2a-4111-993e-4f1891ad9a53,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,00000981-5e2a-4111-993e-4f1891ad9a53,,,61.0,44.0,142.0,NaN
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,2023-07-24 15:03:37,click_bottom_navigation_lab,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,00000a7a-ba72-4332-b4a9-7910670aaeb2,,,119.0,545.0,3287.0,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11441319 entries, 0 to 11441318
Data columns (total 11 columns):
 #   Column          Dtype         
---  ------          -----         
 0   event_id        str           
 1   event_datetime  datetime64[us]
 2   event_key       str           
 3   session_id      str           
 4   id              str           
 5   item_name       str           
 6   page_name       str           
 7   friend_count    float64       
 8   votes_count     float64       
 9   heart_balance   float64       
 10  question_id     float64       
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 2.2 GB


# 중복 체크

In [5]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['session_id'].nunique()

253616

- 전체 중복은 없음.
- 행동 로그 단위의 테이블로 session_id는 중복이 있을 수 밖에 없다고 판단.
따라서, 중복 처리는 따로 진행하지 않는다.

# 결측 체크

In [19]:
df.head()

,event_id,event_datetime,event_key,session_id,id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,2023-07-18 19:40:17,$session_start,4OzYh3seq3VKytpSn5pvQkZNQii1,00000533-3f1c-4b3b-81f1-0c8f35754b4e,,,NaN,NaN,NaN,NaN
1,00000716-27e9-4e72-a602-d0ce61784b06,2023-07-18 21:07:24,click_question_open,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,00000716-27e9-4e72-a602-d0ce61784b06,,,64.0,436.0,4830.0,NaN
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,2023-08-06 20:18:03,click_bottom_navigation_profile,6bcea65d-9f40-46fc-888c-700fe707483f,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,,,26.0,174.0,4729.0,NaN
3,00000981-5e2a-4111-993e-4f1891ad9a53,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,00000981-5e2a-4111-993e-4f1891ad9a53,,,61.0,44.0,142.0,NaN
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,2023-07-24 15:03:37,click_bottom_navigation_lab,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,00000a7a-ba72-4332-b4a9-7910670aaeb2,,,119.0,545.0,3287.0,NaN


In [18]:
df.isna().sum()

event_id                 0
event_datetime           0
event_key                0
session_id               0
id                       0
item_name                0
page_name                0
friend_count        752556
votes_count         754554
heart_balance       728643
question_id       10991835
dtype: int64

- 데이터 프레임상 item_name, page_name에서도 빈 문자열 확인 결측으로 집계되고 있지 않음.

### page_name 체크

In [14]:
df['page_name'].unique()

<ArrowStringArray>
[       '',  'notice',    'home', 'profile',    '학교선택',    '학년선택',     '반선택',
    '번호인증',    '성별선택',   '아이디입력',    '프사설정',  'invite',    '이름입력']
Length: 13, dtype: str

In [20]:
(df['page_name'] == '').sum()

np.int64(10652540)

In [30]:
filter_event_key = df[df['page_name'] == '']['event_key'].unique()

In [31]:
df[df['event_key'].isin(filter_event_key)]['page_name'].unique()

<ArrowStringArray>
['']
Length: 1, dtype: str

In [ ]:
df[~(df['event_key'].isin(filter_event_key))]['event_key'].unique()

<ArrowStringArray>
[ 'click_notice_detail',     'click_attendance',   'click_question_ask',
 'click_question_start',    'click_profile_ask',          'view_signup',
  'click_friend_invite',  'click_invite_friend',         'click_notice']
Length: 9, dtype: str

- 9개의 event_key를 제외한 다른 event_key는 모두 빈 문자열로 데이터가 들어가있으므로 해당 부분은 null값으로 유지될 수 있도록 처리한다.

### item_name

In [37]:
df[df['event_key'] == "click_purchase"]['item_name'].unique()

<ArrowStringArray>
['777 하트', '무료충전소', '1000 하트', '200 하트', '4000 하트']
Length: 5, dtype: str

- item_name의 경우 빈 문자열 발생은 event_key가 "click_purchase"가 아닌 모든 경우 빈 문자열이 발생한다. 따라서 해당 내용도 null값으로 대체하여 유지될 수 있도록 처리한다.

## friend_count 체크

In [48]:
df[df['friend_count'].isna()].shape

(752556, 11)

- friend_count가 결측인 데이터 752,556건 확인

In [50]:
# 결측의 원인이 session_id인가?

users = df[df['friend_count'].isna()]['session_id'].unique()
users

<ArrowStringArray>
[        '4OzYh3seq3VKytpSn5pvQkZNQii1',
         'LztzUUFoRxdqTSPgQrX3MAAyNkM2',
         'NOdvth0cBJV15fIP2sXxMAMEUEr1',
         'qLdDlFGK9qObRuGXK20KAGbqzRZ2',
         'i6toBlZRPDNAnwhvkIWwD16Fcts1',
 '98078b29-9edc-4097-bb7a-500141473bb3',
         'ijJOxCkzhyZ0DJaQYq0JAbCnURO2',
         '3nBBDbvgWaNeqPvsE4YnCQT9ILt2',
 '841DDE93-BBF3-4DFE-8003-CE695AD696FF',
         'Smot2x4FSsNL1rXLWRGWCpbz2Hw2',
 ...
         'oYzJSxKWpfeMJ5Vfl8A81Abdb0J2',
         'WfAsiCiDb5UPKdvM5JwlFxmUNbG2',
         'rj9ILcLEjiS3XWEkkC9yXxhGPnR2',
 '5596edc7-6270-4c3c-ae4c-7385b66dffd2',
         'KehKcSS3HENKnyMgd9Ml76FQTZ13',
         'cK6MxakwE7fmYLNIhvqYnFpuQ6H3',
         'ALLToGjPIiNhvj2g3RDZNuayuYj1',
         'aoNxEWgG20Ry14fC6UP5u7qh9YX2',
 'dc217f9c-f03a-44c5-8647-ab27ed75b87f',
         'ydmCUtVhR7VcWUtjKZ3iHdLAHPz2']
Length: 167435, dtype: str

In [ ]:
# 특정 시기에 결측이 있는가?

print(df[df['friend_count'].isna()]['event_datetime'].min())
print(df[df['friend_count'].isna()]['event_datetime'].max())

2023-07-18 00:00:06
2023-08-10 23:59:57


- 해당데이터의 전체 기간에 해당하는 것으로 특정기간에 발생한 결측은 아닌 것으로 확인

In [40]:
friend_check_event = df[df['friend_count'].isna()]['event_key'].unique()
friend_check_event

<ArrowStringArray>
[      '$session_start',         '$session_end',           'launch_app',
           'view_login',        'view_home_tap',          'view_signup',
               'button',     'view_profile_tap',  'view_friendplus_tap',
    'view_timeline_tap',   'view_questions_tap',  'click_question_open',
 'click_question_share',         'click_notice',  'click_notice_detail']
Length: 15, dtype: str

- friend_count가 결측인 이벤트를 확인해보니 15개의 이벤트가 확인됨.

In [69]:
nnfriend_check_event = df[~(df['friend_count'].isna())]['event_key'].unique()
nnfriend_check_event

<ArrowStringArray>
[              'click_question_open',   'click_bottom_navigation_profile',
                         'view_shop',       'click_bottom_navigation_lab',
                        'launch_app', 'click_bottom_navigation_questions',
                      'view_lab_tap',                     'skip_question',
                'view_questions_tap',                 'view_timeline_tap',
  'click_bottom_navigation_timeline',                    '$session_start',
                      '$session_end',               'click_notice_detail',
          'click_random_ask_shuffle',                 'complete_question',
         'click_appbar_alarm_center',           'click_appbar_chat_rooms',
                  'view_profile_tap',         'click_timeline_chat_start',
                  'click_attendance',                'click_question_ask',
              'click_question_share',          'click_appbar_friend_plus',
              'click_appbar_setting',              'click_question_start',
      

# 이벤트 설명 + 이벤트별 결측데이터 체크

In [72]:
event_desc = {
    '$session_start': '세션 시작',
    '$session_end': '세션 종료',
    'button': '-',
    'click_appbar_alarm_center': '상단 appbar에서 알림 모양 클릭',
    'click_appbar_chat_rooms': '상단 appbar에서 메시지 모양 클릭',
    'click_appbar_friend_plus': '상단 appbar에서 친구 모양 클릭',
    'click_appbar_setting': '-',
    'click_attendance': '출석체크 클릭',
    'click_autoadd_contact': '친구 추천 탭 위의 연락처 자동 친구 추가하기 버튼 클릭',
    'click_bottom_navigation_lab': '하단 네비게이션에서 lab 클릭',
    'click_bottom_navigation_profile': '하단 네비게이션에서 프로필 클릭',
    'click_bottom_navigation_questions': '하단 네비게이션에서 질문 클릭',
    'click_bottom_navigation_timeline': '하단 네비게이션에서 타임라인 클릭',
    'click_community_chat': '채팅 클릭',
    'click_copy_profile_link_ask': 'ask에서 내 프로필 링크 복사하기 클릭',
    'click_copy_profile_link_profile': '본인 프로필 링크 복사하기 클릭',
    'click_friend_invite': '질문 대기상태에서 친구 초대하고 바로 받기 클릭',
    'click_invite_friend': 'ask에서 친구 초대하기 클릭',
    'click_notice': '알림 센터 클릭',
    'click_notice_detail': '알림 센터 디테일에서 특정 알림 클릭',
    'click_profile_ask': '친구 프로필에서 친구에게 글 남기기 버튼 클릭',
    'click_purchase': '구매할 하트 상품 클릭',
    'click_question_ask': '홈화면에서 ask 클릭',
    'click_question_open': '받은 질문을 열 때',
    'click_question_share': '받은 질문을 공유할 때',
    'click_question_start': '홈화면에서 질문 start 클릭',
    'click_random_ask_normal': '기존 타임라인에서 일반적인 ask 클릭',
    'click_random_ask_other': '기존 타임라인에서 다른 친구에게 글 남기기 클릭',
    'click_random_ask_shuffle': '기존 타임라인 상단 ask 랜덤 셔플 클릭',
    'click_timeline_chat_start': '기존 타임라인에서 채팅 클릭',
    'complete_purchase': '하트 구입 완료',
    'complete_question': '질문 완료',
    'complete_signup': '회원가입 완료',
    'launch_app': '앱 실행',
    'skip_question': '질문 스킵',
    'view_friendplus_tap': '친구 추천 화면 진입',
    'view_home_tap': '홈 진입',
    'view_lab_tap': '실험실 탭에 진입',
    'view_login': '로그인 페이지 진입',
    'view_profile_tap': '프로필 화면 진입',
    'view_questions_tap': '질문 화면 진입',
    'view_shop': '하트 충전소 진입',
    'view_signup': '회원가입 페이지 진입',
    'view_timeline_tap': '타임라인 화면 진입'
}

In [84]:
# 결측 패턴을 확인할 컬럼
columns = [
    'friend_count',
    'votes_count',
    'heart_balance'
]

missing_by_event = {}

for col in columns:
    
    result = (
        df.groupby('event_key')
          .agg(
              total_count=(col, 'size'),
              missing_count=(col, lambda x: x.isna().sum()),
              non_missing_count=(col, lambda x: x.notna().sum())
          )
    )

    # 결측률 계산
    result['missing_rate'] = (
        result['missing_count']
        / result['total_count']
    )

    # 이벤트 설명 추가
    result['event_desc'] = result.index.map(event_desc)

    # 이벤트 설명을 첫 번째 컬럼으로 이동
    result.insert(
        0,
        'event_desc',
        result.pop('event_desc')
    )

    # 결측률 기준 정렬
    result = result.sort_values(
        'missing_rate',
        ascending=False
    )

    # 결과 저장
    missing_by_event[col] = result

In [86]:
friend_missing_by_event = missing_by_event['friend_count']
votes_missing_by_event = missing_by_event['votes_count']
heart_missing_by_event = missing_by_event['heart_balance']

In [87]:
friend_missing_by_event[friend_missing_by_event['missing_count'] != 0].sort_values(by='missing_count', ascending=False)

,event_desc,total_count,missing_count,non_missing_count,missing_rate
event_key,,,,,
$session_start,세션 시작,1036852,337709,699143,0.325706
launch_app,앱 실행,986388,201902,784486,0.204688
$session_end,세션 종료,649658,171391,478267,0.263817
view_login,로그인 페이지 진입,49275,18963,30312,0.384840
view_signup,회원가입 페이지 진입,25630,16744,8886,0.653297
view_home_tap,홈 진입,5392,5375,17,0.996847
button,-,428,425,3,0.992991
view_timeline_tap,타임라인 화면 진입,1194508,15,1194493,0.000013
view_questions_tap,질문 화면 진입,353400,14,353386,0.000040


### friend_count 결측치 확인

`friend_count`의 결측이 특정 이벤트에서 발생하는지 확인하기 위해
`event_key`별 전체 로그 수, 결측 수, 비결측 수 및 결측률을 비교하였다.

확인 결과 `friend_count`의 결측은 모든 이벤트에서 동일하게 발생하지 않고,
일부 이벤트에서 상대적으로 높은 비율로 관찰되었다.

- `view_home_tap(홈 진입)`은 대부분의 로그에서 `friend_count`가 결측으로 나타났다.
- `view_signup(회원가입 페이지 진입)`, `view_login(로그인 페이지 진입)`에서도 비교적 높은 결측률이 확인되었다.
- `$session_start`, `$session_end`, `launch_app`에서는 결측과 비결측이 함께 존재하였다.
- 반면 질문, 타임라인, 출석 등 서비스 내부 행동과 관련된 다수의 이벤트에서는 결측이 거의 발생하지 않았다.

따라서 `friend_count`의 결측은 단순한 무작위 결측이라기보다
이벤트의 발생 시점이나 사용자 상태, 로그 수집 방식 등에 따라 발생할 가능성이 있다.

다만 동일 이벤트 내에서도 결측과 비결측이 함께 존재하는 경우가 있으므로,
이벤트 종류만으로 결측 원인을 단정하기는 어렵다.
추가적으로 이벤트 발생 시점 및 다른 속성의 결측 패턴을 확인할 필요가 있다.

단, 현재 친구와 관련된 이벤트 `친구추천 화면 진입`의 경우를 제외한 친구관련 이벤트의 결측은 없는것으로 보이고 결측으로 남겨두는 것이 적합하다고 판단된다.

## votes_count 체크

In [90]:
# 특정 시기에 결측이 있는가?

print(df[df['votes_count'].isna()]['event_datetime'].min())
print(df[df['votes_count'].isna()]['event_datetime'].max())

2023-07-18 00:00:06
2023-08-10 23:59:57


In [88]:
votes_missing_by_event[votes_missing_by_event['missing_count'] != 0].sort_values(by='missing_count', ascending=False)

,event_desc,total_count,missing_count,non_missing_count,missing_rate
event_key,,,,,
$session_start,세션 시작,1036852,338613,698239,0.326578
launch_app,앱 실행,986388,202446,783942,0.205240
$session_end,세션 종료,649658,171858,477800,0.264536
view_login,로그인 페이지 진입,49275,19007,30268,0.385733
view_signup,회원가입 페이지 진입,25630,16766,8864,0.654155
view_home_tap,홈 진입,5392,5392,0,1.000000
button,-,428,425,3,0.992991
view_timeline_tap,타임라인 화면 진입,1194508,15,1194493,0.000013
view_questions_tap,질문 화면 진입,353400,14,353386,0.000040


### votes_count 결측치 확인

`votes_count`의 결측도 동일한 방식으로 확인한 결과.

`votes_count`의 결측 또한, 모든 이벤트에서 동일하게 발생하지 않고,
일부 이벤트에서 상대적으로 높은 비율로 관찰되었다.

- 홈, 앱, 세션시작, 종료 로그인 페이지 등 질문과 관계가 약한 이벤트에서 결측이 높게 나타났다.

단, 결측이 높지만 결측이 아닌 값들 또한 존재하여 해당 이벤트들이 `votes_count`의 값을 불러오지 않는다고는 볼 수 없다.

따라서 `votes_count`의 결측도 발생 시점이나 사용자 상태, 로그 수집 방식 등에 따라 발생할 가능성이 있지만 현재까지는 근거 확인이 되지 않은 상태이다.

## heart_balance 체크

In [91]:
# 특정 시기에 결측이 있는가?

print(df[df['heart_balance'].isna()]['event_datetime'].min())
print(df[df['heart_balance'].isna()]['event_datetime'].max())

2023-07-18 00:00:06
2023-08-10 23:59:57


In [89]:
heart_missing_by_event[heart_missing_by_event['missing_count'] != 0].sort_values(by='missing_count', ascending=False)

,event_desc,total_count,missing_count,non_missing_count,missing_rate
event_key,,,,,
$session_start,세션 시작,1036852,326980,709872,0.315358
launch_app,앱 실행,986388,195336,791052,0.198032
$session_end,세션 종료,649658,165822,483836,0.255245
view_login,로그인 페이지 진입,49275,18352,30923,0.372440
view_signup,회원가입 페이지 진입,25630,16471,9159,0.642645
view_home_tap,홈 진입,5392,5210,182,0.966246
button,-,428,425,3,0.992991
view_timeline_tap,타임라인 화면 진입,1194508,15,1194493,0.000013
view_questions_tap,질문 화면 진입,353400,14,353386,0.000040


### heart_balance 결측치 확인

`heart_balance`의 결측도 동일한 방식으로 확인한 결과.

`heart_balance`의 결측 또한, 모든 이벤트에서 동일하게 발생하지 않고,
일부 이벤트에서 상대적으로 높은 비율로 관찰되었다.

- 홈, 앱, 세션시작, 종료 로그인 페이지 등 질문과 관계가 약한 이벤트에서 결측이 높게 나타났다.

단, 결측이 높지만 결측이 아닌 값들 또한 존재하여 해당 이벤트들이 `heart_balance`의 값을 불러오지 않는다고는 볼 수 없다.

따라서 `heart_balance`의 결측도 발생 시점이나 사용자 상태, 로그 수집 방식 등에 따라 발생할 가능성이 있지만 현재까지는 근거 확인이 되지 않은 상태이다.

## 날짜의 따른 결측인가?

In [92]:
target = df[df['event_key'] == '$session_start'].copy()

target['date'] = target['event_datetime'].dt.date

daily_missing = (
    target.groupby('date')
          .agg(
              total_count=('friend_count', 'size'),
              missing_count=('friend_count', lambda x: x.isna().sum()),
              non_missing_count=('friend_count', lambda x: x.notna().sum())
          )
)

daily_missing['missing_rate'] = (
    daily_missing['missing_count']
    / daily_missing['total_count']
)

daily_missing

,total_count,missing_count,non_missing_count,missing_rate
date,,,,
2023-07-18,67748,32140,35608,0.474405
2023-07-19,49632,16743,32889,0.337343
2023-07-20,71559,32910,38649,0.459900
2023-07-21,72828,31619,41209,0.434160
2023-07-22,45710,12828,32882,0.280639
2023-07-23,60808,24336,36472,0.400210
2023-07-24,41569,10880,30689,0.261734
2023-07-25,36720,9073,27647,0.247086
2023-07-26,34156,8144,26012,0.238435


### 날짜별 결측을 확인한 결과.

- 가장 많은 수의 이벤트를 가졌던 `세션 시작` 기준으로 확인한 결과 결측 비율이 대체적으로 처음에 비해 낮아지는 것처럼 보이지만 중간중간 튀는 값들이 존재. 전체 모수가 늘어남에 따른 변화정도로 읽혀지며, 해당 결측이 특정 기간에 몰리는 현상으로 보기에는 어렵다. 따라서, 특정기간의 오류로인해 결측이 발생했다고 보기 어렵다.

## 세션 아이디에 따른 결측인가?
- 유저별 특성 및 세부 정보 등에 의한 결측인가를 확인

In [95]:
target = df[df['event_key'] == '$session_start'].copy()

user_missing_pattern = (
    target.groupby('session_id')['friend_count']
          .agg(
              event_count='size',
              missing_count=lambda x: x.isna().sum(),
              non_missing_count=lambda x: x.notna().sum()
          )
)

user_missing_pattern['pattern'] = 'mixed'

user_missing_pattern.loc[
    user_missing_pattern['missing_count'] == 0,
    'pattern'
] = 'always_value'

user_missing_pattern.loc[
    user_missing_pattern['non_missing_count'] == 0,
    'pattern'
] = 'always_missing'

user_missing_pattern['pattern'].value_counts()

pattern
always_missing    93837
always_value      86086
mixed             72954
Name: count, dtype: int64

### friend_count 컬럼 대상 세션아이디에 따른 결측 확인 결과
- 지속적으로 결측이 발생하는 세션아이디, 항상 값이 존재하는 세션아이디, 두 경우 모두 해당되는 세션 아이디의 수를 확인한 결과로 동일 사용자에서도 결측과 비결측이 모두 발생하는 사례가 다수 확인되었다.

따라서 특정 사용자에게만 고정적으로 발생하는 결측으로 보기 어려우며,
동일 사용자 내에서도 특정 상태 또는 로그 발생 조건에 따라
결측 여부가 달라질 가능성이 있다.

In [94]:
df[
    ['friend_count', 'votes_count', 'heart_balance']
].isna().value_counts()

friend_count  votes_count  heart_balance
False         False        False            10686765
True          True         True               727097
                           False               25459
False         True         True                 1546
                           False                 452
Name: count, dtype: int64

## 특정 컬럼이 아닌 세 컬럼이 모두 결측인 경우를 확인한 결과

friend_count, votes_count, heart_balance의 결측 여부를 함께 확인한 결과, 대부분의 로그에서 세 컬럼이 모두 존재하거나 모두 결측되는 패턴이 나타났다. 따라서 세 컬럼의 결측은 서로 독립적으로 발생하기보다 공통된 로그 수집 조건 또는 사용자 상태와 관련되어 있을 가능성이 있다. 다만 일부 로그에서는 개별 컬럼만 결측되는 예외 패턴도 존재한다.

# <>

### 사용자 상태 관련 컬럼 결측 탐색 결과

`friend_count`, `votes_count`, `heart_balance` 컬럼에서 다수의 결측이 확인되어,
결측이 특정 이벤트, 발생 시점 또는 사용자에 의해 발생하는지 추가 탐색을 진행하였다.

#### 1. 이벤트별 결측 패턴

`event_key`별 결측률을 확인한 결과,
회원가입·로그인·앱 실행·세션 시작/종료 등 일부 이벤트에서 상대적으로 높은 결측률이 관찰되었다.

반면 질문, 타임라인, 출석 등 서비스 내부 행동과 관련된 다수의 이벤트에서는
결측이 거의 발생하지 않았다.

그러나 `$session_start` 등 동일한 이벤트 내에서도 결측과 비결측이 함께 존재하여,
이벤트 종류만으로 결측 발생 여부를 설명하기는 어려웠다.


#### 2. 발생 시점에 따른 결측 패턴

`$session_start` 이벤트를 기준으로 `friend_count`의 일자별 결측률을 확인하였다.

날짜에 따라 결측률의 차이는 존재했으나,
결측은 로그 수집 기간(2023-07-18 ~ 2023-08-10) 전반에 걸쳐 지속적으로 발생하였다.

특정 시점 이후 결측이 일관되게 발생하거나 해소되는 형태는 확인되지 않아,
특정 기간의 일시적인 수집 문제만으로 결측을 설명하기는 어려웠다.


#### 3. 사용자별 결측 패턴

`$session_start` 이벤트에서 `session_id`를 사용자 식별 기준으로 하여
`friend_count`의 결측 여부를 확인하였다.

- 항상 결측인 사용자: 93,837
- 항상 값이 존재하는 사용자: 86,086
- 결측과 비결측이 모두 존재하는 사용자: 72,954

동일 사용자에서도 결측과 비결측이 모두 발생하는 사례가 다수 확인되었다.

따라서 특정 사용자에게만 고정적으로 발생하는 결측으로 보기 어려우며,
동일 사용자 내에서도 특정 상태 또는 로그 발생 조건에 따라
결측 여부가 달라질 가능성이 있다.


#### 4. 세 컬럼의 동시 결측 여부

`friend_count`, `votes_count`, `heart_balance`의 결측 여부를 함께 비교한 결과,
대부분의 로그는 세 컬럼이 모두 존재하거나 모두 결측되는 형태로 나타났다.

- 세 컬럼 모두 값 존재: 10,686,765건
- 세 컬럼 모두 결측: 727,097건
- 일부 컬럼만 결측: 27,457건

일부 예외는 존재하지만, 결측이 발생한 로그에서는 세 컬럼이 함께 결측되는 패턴이
두드러지게 나타났다.

따라서 세 컬럼의 결측이 각각 독립적인 원인으로 발생하기보다는
공통된 사용자 상태 또는 로그 수집 조건과 관련되어 있을 가능성이 있다고 판단하였다.


#### 5. 현재까지의 판단

현재까지의 탐색 결과만으로 결측의 정확한 원인을 특정하기는 어렵다.

다만,

- 특정 이벤트만의 문제로 설명하기 어려움
- 특정 기간의 일시적인 문제로 설명하기 어려움
- 특정 사용자만의 문제로 설명하기 어려움
- 세 사용자 상태 관련 컬럼이 함께 결측되는 강한 패턴이 존재함

을 확인하였다.

따라서 현 단계에서는 해당 결측을 단순한 데이터 오류로 판단하여
삭제하거나 임의의 값으로 대체하지 않고 유지한다.

이후 세 컬럼이 모두 결측인 로그와 값이 존재하는 로그의 사용자 상태 및
행동 특성을 추가 비교하여 공통 결측이 발생하는 조건을 탐색할 예정이다.